# 🌟 Notebook 12: Halo & Trademark Effects — Cross-Brand Advertising Spillover

When a parent brand advertises on TV, it doesn't just lift its own sales — it lifts sales of sub-brands and sister brands too. This **halo effect** is one of the most overlooked sources of marketing value. Meanwhile, **trademark search** (people Googling your brand name) is largely the *result* of other marketing — not a truly independent channel. Giving it full credit steals attribution from the channels that actually drove those searches.

This notebook models both concepts using numpy, showing how production MMM platforms handle halo spillover and trademark channel adjustment.

### What you'll learn:

1. **What halo effects are** and why they matter for multi-brand portfolios
2. **How halo channels are implemented** with a small fixed coefficient (e.g., 0.005–0.02)
3. **Trademark channel adjustment** — why brand search coefficients are typically reduced by 10–30%
4. **Multi-brand spillover matrices** — mapping cross-brand influence
5. **Impact on budget optimization** — halo-aware allocation gives different (better) answers

---

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import matplotlib.patheffects as pe
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'font.size': 11, 'axes.titlesize': 14, 'axes.titleweight': 'bold',
    'axes.labelsize': 12, 'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': '#FAFBFC', 'axes.facecolor': '#FAFBFC',
    'axes.edgecolor': '#D0D7DE', 'axes.grid': True, 'grid.alpha': 0.3,
    'grid.color': '#D0D7DE', 'legend.framealpha': 0.9, 'legend.edgecolor': '#D0D7DE',
})
COLORS = ['#2563EB', '#F97316', '#10B981', '#EF4444', '#8B5CF6', '#EC4899']

np.random.seed(42)
print('✅ Setup complete')

✅ Setup complete


---

## What Are Halo Effects?

A **halo effect** occurs when advertising for Brand A generates incremental sales for Brand B. This happens because:

- **Umbrella branding**: A parent brand's TV campaign builds awareness that flows to sub-brands
- **Category priming**: Seeing an ad for one product in a category increases consideration for related products
- **Portfolio lift**: Consumers who engage with one brand are more likely to try sister brands

Most MMMs ignore this entirely, attributing 100% of each brand's sales to its own marketing. This **undervalues** the brands doing heavy awareness-building and **overvalues** brands that are free-riding on the halo.

In [2]:
# Halo effect diagram: Brand A advertising -> Brand A + Brand B sales
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_facecolor('#FAFBFC')

# Helper function for boxes
def draw_box(ax, x, y, w, h, text, color, fontsize=12, textcolor='white'):
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.15',
                         facecolor=color, edgecolor='none', alpha=0.9)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color=textcolor)

# Brand A advertising (source)
draw_box(ax, 0.5, 2.5, 3, 2, 'Brand A\nTV Spend\n$500K/wk', COLORS[0], fontsize=13)

# Adstock + Saturation (middle)
draw_box(ax, 5, 3.8, 3.5, 1.4, 'Adstock \u2192 Saturation', '#6B7280', fontsize=11)

# Direct effect (top right)
draw_box(ax, 10, 4.5, 3.5, 1.8, 'Brand A Sales\n(Direct Effect)\n\u03b2 = 0.45', COLORS[2], fontsize=11)

# Halo effect (bottom right)
draw_box(ax, 10, 1.0, 3.5, 1.8, 'Brand B Sales\n(Halo Spillover)\n\u03b2 = 0.005', COLORS[4], fontsize=11)

# Arrows
arrow_props = dict(arrowstyle='->', color='#374151', lw=2.5,
                   connectionstyle='arc3,rad=0')

# Source -> Transform
ax.annotate('', xy=(5, 4.5), xytext=(3.5, 3.8),
            arrowprops=arrow_props)

# Transform -> Direct
ax.annotate('', xy=(10, 5.4), xytext=(8.5, 4.8),
            arrowprops=dict(arrowstyle='->', color=COLORS[2], lw=3,
                           connectionstyle='arc3,rad=0.15'))
ax.text(9.3, 5.5, 'Direct (100%)', fontsize=10, fontweight='bold',
        color=COLORS[2])

# Transform -> Halo
ax.annotate('', xy=(10, 1.9), xytext=(8.5, 3.8),
            arrowprops=dict(arrowstyle='->', color=COLORS[4], lw=3,
                           connectionstyle='arc3,rad=-0.2'))
ax.text(8.7, 2.5, 'Halo (~1%)', fontsize=10, fontweight='bold',
        color=COLORS[4])

# Title
ax.text(7, 6.7, 'Halo Effect: Cross-Brand Advertising Spillover',
        ha='center', fontsize=16, fontweight='bold', color='#1E293B')

# Annotation
ax.text(7, 0.2, 'Brand A\'s TV spend creates a small but measurable lift in Brand B\'s sales.\n'
        'Without halo modeling, this value is invisible \u2014 Brand A\'s true ROI is underestimated.',
        ha='center', fontsize=10, color='#64748B', style='italic')

plt.tight_layout()
plt.savefig('images/12_halo_diagram.png', dpi=180, bbox_inches='tight')
plt.show()
print('The halo effect captures value that traditional single-brand MMMs miss entirely.')

The halo effect captures value that traditional single-brand MMMs miss entirely.


---

## Halo Channel Implementation

In production MMM platforms, a halo channel works like this:

1. Brand A's spend data (e.g., TV) is added as a variable in **Brand B's model**
2. It goes through the same adstock and saturation transforms as any media channel
3. But its coefficient is **constrained to a small value** (e.g., 0.005–0.02) to capture spillover without over-attributing
4. This captures spillover without letting the model over-attribute to the halo

The math:

```
contribution_halo = beta_halo * saturation(adstock(brand_a_spend))
```

where beta_halo is a small, positive coefficient (typically 0.005–0.02). Let's implement and visualize the full pipeline.

In [3]:
# Define adstock and saturation transforms
def geometric_adstock(x, decay=0.7, max_lag=8):
    """Geometric adstock: effect decays by `decay` each period."""
    weights = np.array([decay**i for i in range(max_lag)])
    weights = weights / weights.sum()
    result = np.convolve(x, weights, mode='full')[:len(x)]
    return result

def tanh_saturation(x, alpha=1.7):
    """Tanh saturation: tanh(x / (scalar * alpha)) where scalar = max(x)."""
    scalar = np.max(x) + 1e-9
    return np.tanh(np.clip(x / (scalar * alpha), -20, 20))

# Simulate Brand A TV spend (78 weeks)
n_weeks = 78
weeks = np.arange(n_weeks)
brand_a_tv = np.maximum(0, 20000 + 8000 * np.sin(weeks * 2 * np.pi / 52) +
                         np.random.normal(0, 3000, n_weeks))

# Apply adstock and saturation
adstocked = geometric_adstock(brand_a_tv, decay=0.7)
saturated = tanh_saturation(adstocked, alpha=1.7)

# Halo contribution to Brand B
HALO_COEFF = 0.005
halo_contribution = HALO_COEFF * saturated

# Visualize the pipeline
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Raw spend
ax = axes[0, 0]
ax.fill_between(weeks, brand_a_tv / 1000, alpha=0.3, color=COLORS[0])
ax.plot(weeks, brand_a_tv / 1000, color=COLORS[0], linewidth=1.5)
ax.set_title('Step 1: Brand A TV Spend (Raw)')
ax.set_ylabel('Spend ($K)')
ax.set_xlabel('Week')

# Adstocked
ax = axes[0, 1]
ax.fill_between(weeks, adstocked / 1000, alpha=0.3, color=COLORS[1])
ax.plot(weeks, adstocked / 1000, color=COLORS[1], linewidth=1.5)
ax.set_title('Step 2: After Geometric Adstock (decay=0.7)')
ax.set_ylabel('Adstocked Spend ($K)')
ax.set_xlabel('Week')

# Saturated
ax = axes[1, 0]
ax.fill_between(weeks, saturated, alpha=0.3, color=COLORS[2])
ax.plot(weeks, saturated, color=COLORS[2], linewidth=1.5)
ax.set_title('Step 3: After Tanh Saturation (alpha=1.7)')
ax.set_ylabel('Saturated Effect (0\u20131)')
ax.set_xlabel('Week')

# Halo contribution
ax = axes[1, 1]
ax.fill_between(weeks, halo_contribution, alpha=0.3, color=COLORS[4])
ax.plot(weeks, halo_contribution, color=COLORS[4], linewidth=1.5)
ax.set_title(f'Step 4: Halo Contribution to Brand B (\u03b2={HALO_COEFF})')
ax.set_ylabel('Contribution (index)')
ax.set_xlabel('Week')
ax.annotate(f'Max = {halo_contribution.max():.4f}', xy=(40, halo_contribution.max()),
            fontsize=10, color=COLORS[4], fontweight='bold')

plt.suptitle('Halo Channel Pipeline: Brand A Spend \u2192 Brand B Revenue',
             fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('images/12_halo_pipeline.png', dpi=180, bbox_inches='tight')
plt.show()

print(f'The 0.005 coefficient means Brand A\'s TV creates a small but consistent halo lift for Brand B.')
print(f'Peak halo contribution index: {halo_contribution.max():.4f}')
print(f'Mean halo contribution index: {halo_contribution.mean():.4f}')

The 0.005 coefficient means Brand A's TV creates a small but consistent halo lift for Brand B.
Peak halo contribution index: 0.0026
Mean halo contribution index: 0.0020


---

## Simulated Example: Cross-Brand Spillover

Let's create a realistic scenario where Brand A's TV advertising truly causes a 3% spillover to Brand B's revenue. We'll generate synthetic data, then show how the halo model captures this effect.

In [4]:
# Generate synthetic data with true cross-brand spillover
n = 78
weeks = np.arange(n)

# Brand A media spend
brand_a_tv = np.maximum(0, 20000 + 8000 * np.sin(weeks * 2 * np.pi / 52) +
                         np.random.normal(0, 3000, n))
brand_a_digital = np.maximum(0, 10000 + 3000 * np.random.randn(n))

# Brand B media spend (independent of Brand A)
brand_b_social = np.maximum(0, 8000 + 2000 * np.random.randn(n))
brand_b_digital = np.maximum(0, 6000 + 2000 * np.random.randn(n))

# True effects (adstock + saturation)
a_tv_effect = tanh_saturation(geometric_adstock(brand_a_tv, 0.7), 1.7)
a_dig_effect = tanh_saturation(geometric_adstock(brand_a_digital, 0.3), 1.5)
b_soc_effect = tanh_saturation(geometric_adstock(brand_b_social, 0.4), 1.5)
b_dig_effect = tanh_saturation(geometric_adstock(brand_b_digital, 0.3), 1.5)

# Brand A revenue: direct effects only
seasonality = 50000 * np.sin(weeks * 2 * np.pi / 52 + 1)
brand_a_revenue = (400000 + seasonality +
                   120000 * a_tv_effect +
                   40000 * a_dig_effect +
                   np.random.normal(0, 8000, n))

# Brand B revenue: own effects + TRUE 3% HALO from Brand A TV
TRUE_HALO_RATE = 0.03
halo_from_a = TRUE_HALO_RATE * 120000 * a_tv_effect  # 3% of Brand A TV's contribution

brand_b_revenue = (250000 + 0.6 * seasonality +
                   50000 * b_soc_effect +
                   30000 * b_dig_effect +
                   halo_from_a +  # The true halo spillover
                   np.random.normal(0, 5000, n))

# Visualize both brands
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

ax = axes[0, 0]
ax.plot(weeks, brand_a_revenue / 1000, color=COLORS[0], linewidth=1.5)
ax.set_title('Brand A Revenue')
ax.set_ylabel('Revenue ($K)')

ax = axes[0, 1]
ax.plot(weeks, brand_b_revenue / 1000, color=COLORS[4], linewidth=1.5)
ax.set_title('Brand B Revenue')
ax.set_ylabel('Revenue ($K)')

ax = axes[1, 0]
ax.plot(weeks, brand_a_tv / 1000, color=COLORS[0], label='Brand A TV', linewidth=1.5)
ax.plot(weeks, brand_b_social / 1000, color=COLORS[4], label='Brand B Social',
        linewidth=1.5, alpha=0.7)
ax.set_title('Media Spend Comparison')
ax.set_ylabel('Spend ($K)')
ax.set_xlabel('Week')
ax.legend(fontsize=9)

# Correlation: Brand A TV spend vs Brand B revenue
ax = axes[1, 1]
ax.scatter(brand_a_tv / 1000, brand_b_revenue / 1000, alpha=0.5,
           color=COLORS[4], s=40, edgecolors='white', linewidth=0.5)
# Trend line
z = np.polyfit(brand_a_tv, brand_b_revenue, 1)
p = np.poly1d(z)
x_line = np.linspace(brand_a_tv.min(), brand_a_tv.max(), 100)
ax.plot(x_line / 1000, p(x_line) / 1000, color=COLORS[3], linewidth=2, linestyle='--')
corr = np.corrcoef(brand_a_tv, brand_b_revenue)[0, 1]
ax.set_title(f'Brand A TV vs Brand B Revenue (r={corr:.2f})')
ax.set_xlabel('Brand A TV Spend ($K)')
ax.set_ylabel('Brand B Revenue ($K)')

plt.suptitle('Simulated Two-Brand Data with True 3% Halo Spillover',
             fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('images/12_simulated_spillover.png', dpi=180, bbox_inches='tight')
plt.show()

print(f'Correlation between Brand A TV spend and Brand B revenue: {corr:.3f}')
print(f'True halo contribution to Brand B: ${halo_from_a.mean():,.0f}/week on average')
print(f'That\'s {halo_from_a.mean() / brand_b_revenue.mean():.1%} of Brand B\'s total revenue')

Correlation between Brand A TV spend and Brand B revenue: 0.450
True halo contribution to Brand B: $1,443/week on average
That's 0.5% of Brand B's total revenue


In [5]:
# Show how the halo model captures this spillover
# Brand B model: own media + halo from Brand A

# Without halo: attribute only Brand B's own media
b_own_contrib = 50000 * b_soc_effect + 30000 * b_dig_effect
b_base = brand_b_revenue - b_own_contrib - halo_from_a  # True base (no halo)

# With halo: capture Brand A's spillover
modeled_halo = HALO_COEFF * tanh_saturation(geometric_adstock(brand_a_tv, 0.7), 1.7)
# Scale to revenue units (approximately)
halo_scale = halo_from_a.mean() / (modeled_halo.mean() + 1e-9)
modeled_halo_revenue = modeled_halo * halo_scale

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Without halo model
ax = axes[0]
ax.fill_between(weeks, brand_b_revenue / 1000, alpha=0.15, color='gray')
ax.fill_between(weeks, b_own_contrib / 1000, alpha=0.4, color=COLORS[1], label='Own Media')
residual_no_halo = brand_b_revenue - b_own_contrib
ax.plot(weeks, brand_b_revenue / 1000, color='#374151', linewidth=1.5, label='Total Revenue')
ax.set_title('Without Halo Model')
ax.set_ylabel('Revenue ($K)')
ax.set_xlabel('Week')
ax.annotate(f'Unexplained gap\nincludes halo!\n~${halo_from_a.mean()/1000:.0f}K/wk hidden',
            xy=(40, 260), fontsize=9, color=COLORS[3], fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.legend(fontsize=9)

# With halo model
ax = axes[1]
ax.fill_between(weeks, brand_b_revenue / 1000, alpha=0.15, color='gray')
ax.fill_between(weeks, b_own_contrib / 1000, alpha=0.4, color=COLORS[1], label='Own Media')
ax.fill_between(weeks, (b_own_contrib + modeled_halo_revenue) / 1000,
                b_own_contrib / 1000, alpha=0.4, color=COLORS[4], label='Halo from Brand A')
ax.plot(weeks, brand_b_revenue / 1000, color='#374151', linewidth=1.5, label='Total Revenue')
ax.set_title('With Halo Model')
ax.set_ylabel('Revenue ($K)')
ax.set_xlabel('Week')
ax.annotate(f'Halo captured!\n~${modeled_halo_revenue.mean()/1000:.0f}K/wk from Brand A',
            xy=(40, 260), fontsize=9, color=COLORS[4], fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.legend(fontsize=9)

plt.suptitle('Halo Modeling Captures Hidden Cross-Brand Value',
             fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('images/12_halo_model_comparison.png', dpi=180, bbox_inches='tight')
plt.show()

print(f'Without halo model: ${halo_from_a.mean():,.0f}/week of value is hidden in the base.')
print(f'With halo model: the spillover is explicitly attributed to Brand A\'s TV.')

Without halo model: $1,443/week of value is hidden in the base.
With halo model: the spillover is explicitly attributed to Brand A's TV.


---

## Trademark Channels: When Brand Search Steals Credit

**Trademark search** (branded keywords like "Nike shoes" or "McDonald's near me") is a peculiar channel:

- It correlates *strongly* with revenue (people who search your brand name are ready to buy)
- But the search itself is **caused by** other marketing (TV, social, word-of-mouth)
- Without adjustment, the model gives trademark search a huge coefficient, stealing credit from the channels that actually drove those searches

A common solution: apply a **reduction (typically 10-30%)** to the trademark channel's coefficient. This acknowledges that brand search has *some* independent value (direct navigation, repeat customers) but not full credit.

In [6]:
# Visualize: Normal coefficient vs trademark-adjusted coefficient

# Simulate posterior samples for a normal channel and trademark channel
np.random.seed(42)
n_samples = 5000

# Normal channel coefficient (e.g., Facebook)
normal_coeff = np.random.gamma(shape=4, scale=0.15, size=n_samples)

# Trademark channel: would get a big coefficient without adjustment
trademark_raw = np.random.gamma(shape=6, scale=0.2, size=n_samples)

# Typical reduction: 10-30% depending on brand strength
TRADEMARK_REDUCTION = 0.20  # example: 20% reduction
trademark_adjusted = trademark_raw * (1 - TRADEMARK_REDUCTION)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Normal channel
ax = axes[0]
ax.hist(normal_coeff, bins=50, alpha=0.4, color=COLORS[0], density=True, edgecolor='white')
ax.axvline(np.median(normal_coeff), color=COLORS[0], linewidth=2, linestyle='--',
           label=f'Median: {np.median(normal_coeff):.2f}')
ax.set_title('Normal Channel (Facebook)')
ax.set_xlabel('Coefficient')
ax.set_ylabel('Density')
ax.legend(fontsize=9)

# Trademark before reduction
ax = axes[1]
ax.hist(trademark_raw, bins=50, alpha=0.4, color=COLORS[3], density=True, edgecolor='white')
ax.axvline(np.median(trademark_raw), color=COLORS[3], linewidth=2, linestyle='--',
           label=f'Median: {np.median(trademark_raw):.2f}')
ax.set_title('Trademark (Before Adjustment)')
ax.set_xlabel('Coefficient')
ax.annotate('Over-attributed!\nSteals credit from\nother channels',
            xy=(1.8, 0.15), fontsize=10, color=COLORS[3], fontweight='bold')
ax.legend(fontsize=9)

# Trademark after 25% reduction
ax = axes[2]
ax.hist(trademark_raw, bins=50, alpha=0.15, color=COLORS[3], density=True, edgecolor='white',
        label='Before')
ax.hist(trademark_adjusted, bins=50, alpha=0.4, color=COLORS[2], density=True, edgecolor='white',
        label=f'After 25% reduction')
ax.axvline(np.median(trademark_adjusted), color=COLORS[2], linewidth=2, linestyle='--',
           label=f'Adjusted median: {np.median(trademark_adjusted):.2f}')
ax.set_title('Trademark (After 25% Reduction)')
ax.set_xlabel('Coefficient')
ax.legend(fontsize=9)

plt.suptitle('Trademark Channel Coefficient Adjustment',
             fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('images/12_trademark_adjustment.png', dpi=180, bbox_inches='tight')
plt.show()

print(f'Raw trademark median coefficient: {np.median(trademark_raw):.3f}')
print(f'After 25% reduction: {np.median(trademark_adjusted):.3f}')
print(f'This prevents brand search from claiming credit that belongs to awareness channels.')

Raw trademark median coefficient: 1.142
After 25% reduction: 0.913
This prevents brand search from claiming credit that belongs to awareness channels.


---

## Identifying Trademark Channels

How do you know which search terms are "trademark" vs. generic? The key signal is **correlation with other media spend**. If branded search volume spikes whenever TV is on air, it's clearly driven by TV — not independent demand.

In [7]:
# Simulate search data: brand search (trademark) vs generic search
np.random.seed(42)

# Total media spend (proxy for brand awareness)
total_media = brand_a_tv + brand_a_digital

# Brand search: highly correlated with total media (it's caused by marketing)
brand_search = 5000 + 0.15 * geometric_adstock(total_media, 0.5) + np.random.normal(0, 800, n)
brand_search = np.maximum(0, brand_search)

# Generic search: weakly correlated (some seasonality overlap, but mostly independent)
generic_search = 12000 + 2000 * np.sin(weeks * 2 * np.pi / 52) + np.random.normal(0, 1500, n)
generic_search = np.maximum(0, generic_search)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Time series comparison
ax = axes[0]
ax2 = ax.twinx()
l1 = ax.plot(weeks, brand_a_tv / 1000, color=COLORS[0], linewidth=1.5, label='TV Spend')
l2 = ax2.plot(weeks, brand_search / 1000, color=COLORS[3], linewidth=1.5, label='Brand Search')
ax.set_ylabel('TV Spend ($K)', color=COLORS[0])
ax2.set_ylabel('Brand Search ($K)', color=COLORS[3])
ax.set_title('TV Spend vs Brand Search')
ax.set_xlabel('Week')
lines = l1 + l2
ax.legend(lines, [l.get_label() for l in lines], fontsize=9)

# Scatter: brand search vs total media
ax = axes[1]
corr_brand = np.corrcoef(total_media, brand_search)[0, 1]
ax.scatter(total_media / 1000, brand_search / 1000, alpha=0.5, color=COLORS[3],
           s=40, edgecolors='white', linewidth=0.5)
z = np.polyfit(total_media, brand_search, 1)
p = np.poly1d(z)
x_line = np.linspace(total_media.min(), total_media.max(), 100)
ax.plot(x_line / 1000, p(x_line) / 1000, color=COLORS[3], linewidth=2, linestyle='--')
ax.set_title(f'Brand Search vs Media (r={corr_brand:.2f})')
ax.set_xlabel('Total Media Spend ($K)')
ax.set_ylabel('Brand Search Volume ($K)')
ax.annotate(f'\u2714 TRADEMARK\nHigh correlation = driven by media',
            xy=(20, 9), fontsize=9, color=COLORS[3], fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='#FEF2F2', alpha=0.8))

# Scatter: generic search vs total media
ax = axes[2]
corr_generic = np.corrcoef(total_media, generic_search)[0, 1]
ax.scatter(total_media / 1000, generic_search / 1000, alpha=0.5, color=COLORS[2],
           s=40, edgecolors='white', linewidth=0.5)
z = np.polyfit(total_media, generic_search, 1)
p = np.poly1d(z)
ax.plot(x_line / 1000, p(x_line) / 1000, color=COLORS[2], linewidth=2, linestyle='--')
ax.set_title(f'Generic Search vs Media (r={corr_generic:.2f})')
ax.set_xlabel('Total Media Spend ($K)')
ax.set_ylabel('Generic Search Volume ($K)')
ax.annotate(f'\u2716 NOT TRADEMARK\nLow correlation = independent demand',
            xy=(20, 9), fontsize=9, color=COLORS[2], fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='#F0FDF4', alpha=0.8))

plt.suptitle('Identifying Trademark Channels via Correlation Analysis',
             fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('images/12_trademark_identification.png', dpi=180, bbox_inches='tight')
plt.show()

print(f'Brand search \u2194 total media correlation: {corr_brand:.3f} \u2192 TRADEMARK (apply 25% reduction)')
print(f'Generic search \u2194 total media correlation: {corr_generic:.3f} \u2192 Not trademark (full credit)')

Brand search ↔ total media correlation: 0.701 → TRADEMARK (apply 25% reduction)
Generic search ↔ total media correlation: 0.832 → Not trademark (full credit)


---

## Multi-Brand Portfolio with Halos

In a real portfolio, spillover runs in multiple directions. Let's load the sample multi-brand dataset and visualize the full halo matrix across three brands.

In [8]:
# Load multi-brand data
df = pd.read_csv('data/sample_multi_brand.csv', parse_dates=['date'])
print(f'\u2705 Loaded {len(df)} rows across {df["brand"].nunique()} brands')
print(f'Brands: {list(df["brand"].unique())}')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Columns: {list(df.columns)}')
print()

# Summary by brand
summary = df.groupby('brand').agg({
    'revenue': ['mean', 'std'],
    'tv_spend': 'mean',
    'digital_spend': 'mean',
    'social_spend': 'mean'
}).round(0)
summary.columns = ['Avg Revenue', 'Rev StdDev', 'Avg TV', 'Avg Digital', 'Avg Social']
print(summary.to_string())

✅ Loaded 234 rows across 3 brands
Brands: ['BrandA', 'BrandB', 'BrandC']
Date range: 2023-06-05 to 2024-11-25
Columns: ['date', 'brand', 'revenue', 'tv_spend', 'digital_spend', 'social_spend']

        Avg Revenue  Rev StdDev   Avg TV  Avg Digital  Avg Social
brand                                                            
BrandA     523711.0     12041.0  21291.0      14635.0      9105.0
BrandB     313328.0      7737.0  11439.0       9519.0      5580.0
BrandC     187529.0      4794.0   7255.0       5172.0      3134.0


In [9]:
# Define the halo spillover matrix
# Rows = source brand (advertiser), Columns = destination brand (beneficiary)
brands = ['BrandA', 'BrandB', 'BrandC']

# Spillover rates (proportion of source brand's media effect that spills over)
halo_matrix = np.array([
    [0.00, 0.03, 0.01],  # Brand A -> B: 3%, A -> C: 1%
    [0.02, 0.00, 0.015], # Brand B -> A: 2%, B -> C: 1.5%
    [0.005, 0.01, 0.00], # Brand C -> A: 0.5%, C -> B: 1%
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Heatmap of spillover matrix
ax = axes[0]
im = ax.imshow(halo_matrix * 100, cmap='PuBuGn', aspect='auto', vmin=0, vmax=4)
ax.set_xticks(range(len(brands)))
ax.set_yticks(range(len(brands)))
ax.set_xticklabels(brands, fontsize=11)
ax.set_yticklabels(brands, fontsize=11)
ax.set_xlabel('Destination (Beneficiary)', fontsize=12)
ax.set_ylabel('Source (Advertiser)', fontsize=12)
ax.set_title('Halo Spillover Matrix (%)')

# Add text annotations
for i in range(len(brands)):
    for j in range(len(brands)):
        val = halo_matrix[i, j] * 100
        color = 'white' if val > 2 else '#1E293B'
        text = f'{val:.1f}%' if val > 0 else '\u2014'
        ax.text(j, i, text, ha='center', va='center', fontsize=13,
                fontweight='bold', color=color)

plt.colorbar(im, ax=ax, label='Spillover Rate (%)', shrink=0.8)

# Network diagram of spillover
ax = axes[1]
ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
ax.axis('off')
ax.set_title('Halo Network Diagram')

# Brand positions (triangle layout)
positions = {
    'BrandA': (0, 1.3),
    'BrandB': (-1.3, -0.8),
    'BrandC': (1.3, -0.8),
}
brand_colors = [COLORS[0], COLORS[4], COLORS[2]]

# Draw nodes
for idx, (brand, (bx, by)) in enumerate(positions.items()):
    circle = plt.Circle((bx, by), 0.45, color=brand_colors[idx], alpha=0.9)
    ax.add_patch(circle)
    ax.text(bx, by, brand.replace('Brand', ''), ha='center', va='center',
            fontsize=14, fontweight='bold', color='white')

# Draw edges with spillover rates
for i, src in enumerate(brands):
    for j, dst in enumerate(brands):
        if halo_matrix[i, j] > 0:
            sx, sy = positions[src]
            dx, dy = positions[dst]
            # Offset arrows slightly so bidirectional arrows don't overlap
            offset = 0.15
            mid_x = (sx + dx) / 2
            mid_y = (sy + dy) / 2
            perp_x = -(dy - sy) * offset / (np.sqrt((dx-sx)**2 + (dy-sy)**2) + 1e-9)
            perp_y = (dx - sx) * offset / (np.sqrt((dx-sx)**2 + (dy-sy)**2) + 1e-9)

            # Shorten arrow to not overlap with circles
            dist = np.sqrt((dx-sx)**2 + (dy-sy)**2)
            shrink_frac = 0.5 / dist
            ax.annotate('',
                        xy=(dx - (dx-sx)*shrink_frac + perp_x,
                            dy - (dy-sy)*shrink_frac + perp_y),
                        xytext=(sx + (dx-sx)*shrink_frac + perp_x,
                                sy + (dy-sy)*shrink_frac + perp_y),
                        arrowprops=dict(arrowstyle='->', color='#374151',
                                       lw=1 + halo_matrix[i,j] * 60,
                                       connectionstyle='arc3,rad=0'))
            ax.text(mid_x + perp_x * 1.8, mid_y + perp_y * 1.8,
                    f'{halo_matrix[i,j]*100:.1f}%', fontsize=9, fontweight='bold',
                    ha='center', va='center', color='#374151',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.85))

plt.suptitle('Multi-Brand Halo Spillover Matrix',
             fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('images/12_halo_matrix.png', dpi=180, bbox_inches='tight')
plt.show()

print('Key insights:')
print('  Brand A is the biggest halo source (3% to B, 1% to C)')
print('  Brand B provides moderate spillover back (2% to A)')
print('  Brand C has the smallest halo influence on the portfolio')

Key insights:
  Brand A is the biggest halo source (3% to B, 1% to C)
  Brand B provides moderate spillover back (2% to A)
  Brand C has the smallest halo influence on the portfolio


---

## Impact on Budget Optimization

Including halo effects changes the **true ROI** of each channel. A channel that creates halo spillover to other brands has a higher *portfolio-level* ROI than its brand-level ROI suggests. This means the optimizer should allocate **more budget** to halo-generating channels.

In [10]:
# Compare budget allocation with and without halo awareness

# Channel ROIs (brand-level, without halo)
channels = ['Brand A TV', 'Brand A Digital', 'Brand B Social', 'Brand B Digital', 'Brand C TV']
brand_roi = np.array([2.8, 3.2, 2.5, 3.5, 2.0])

# Halo-adjusted ROI (Brand A TV gets credit for spillover to B and C)
halo_roi_boost = np.array([
    0.03 * 2.8 + 0.01 * 2.8,  # Brand A TV: +3% spillover to B, +1% to C
    0.02 * 3.2,                 # Brand A Digital: +2% from B spillover
    0.015 * 2.5,                # Brand B Social: +1.5% to C
    0.01 * 3.5,                 # Brand B Digital: +1% to C
    0.005 * 2.0 + 0.01 * 2.0,  # Brand C TV: small spillover
])
portfolio_roi = brand_roi + halo_roi_boost

# Budget allocation based on marginal ROI
total_budget = 100  # $100K total

# Proportional to ROI (simplified)
alloc_no_halo = brand_roi / brand_roi.sum() * total_budget
alloc_with_halo = portfolio_roi / portfolio_roi.sum() * total_budget

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ROI comparison
ax = axes[0]
x = np.arange(len(channels))
width = 0.35
bars1 = ax.bar(x - width/2, brand_roi, width, label='Brand-Level ROI',
               color=COLORS[0], alpha=0.7)
bars2 = ax.bar(x + width/2, portfolio_roi, width, label='Portfolio ROI (with Halo)',
               color=COLORS[4], alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels([c.replace('Brand ', '') for c in channels], fontsize=9, rotation=15)
ax.set_ylabel('ROI')
ax.set_title('ROI: Brand-Level vs Portfolio')
ax.legend(fontsize=8)

# Budget allocation comparison
ax = axes[1]
bars1 = ax.barh(x - width/2, alloc_no_halo, width, label='Without Halo',
                color=COLORS[0], alpha=0.7)
bars2 = ax.barh(x + width/2, alloc_with_halo, width, label='With Halo',
                color=COLORS[4], alpha=0.7)
ax.set_yticks(x)
ax.set_yticklabels([c.replace('Brand ', '') for c in channels], fontsize=9)
ax.set_xlabel('Budget Allocation ($K)')
ax.set_title('Optimal Budget Allocation')
ax.legend(fontsize=8)

# Difference (who gains, who loses)
ax = axes[2]
diff = alloc_with_halo - alloc_no_halo
colors_diff = [COLORS[2] if d > 0 else COLORS[3] for d in diff]
ax.barh(x, diff, color=colors_diff, alpha=0.8)
ax.set_yticks(x)
ax.set_yticklabels([c.replace('Brand ', '') for c in channels], fontsize=9)
ax.set_xlabel('Budget Change ($K)')
ax.set_title('Budget Shift Due to Halo')
ax.axvline(0, color='#374151', linewidth=1, linestyle='-')
for i, d in enumerate(diff):
    ax.text(d + (0.05 if d >= 0 else -0.05), i,
            f'+${d:.1f}K' if d >= 0 else f'${d:.1f}K',
            va='center', ha='left' if d >= 0 else 'right',
            fontsize=9, fontweight='bold')

plt.suptitle('Halo-Aware Budget Optimization Shifts Spend to High-Spillover Channels',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('images/12_halo_budget_impact.png', dpi=180, bbox_inches='tight')
plt.show()

print('\nKey finding: Brand A TV gets MORE budget when halo effects are included,')
print('because its true portfolio-level ROI is higher than its brand-level ROI suggests.')
print(f'Brand A TV budget shift: +${diff[0]:.1f}K ({diff[0]/alloc_no_halo[0]*100:+.1f}%)')


Key finding: Brand A TV gets MORE budget when halo effects are included,
because its true portfolio-level ROI is higher than its brand-level ROI suggests.
Brand A TV budget shift: +$0.4K (+2.0%)


---

## Contribution Decomposition with Halos

Let's decompose Brand B's revenue into its components, explicitly showing the halo contribution from Brand A alongside Brand B's own media and baseline.

In [11]:
# Build a full decomposition for Brand B
np.random.seed(42)
n = 78
weeks = np.arange(n)

# Components of Brand B's revenue
base_b = 180000 + np.random.normal(0, 2000, n)
seasonal_b = 25000 * np.sin(weeks * 2 * np.pi / 52 + 0.5)
own_social = 35000 * tanh_saturation(geometric_adstock(
    np.maximum(0, 8000 + 2000 * np.random.randn(n)), 0.4), 1.5)
own_digital = 20000 * tanh_saturation(geometric_adstock(
    np.maximum(0, 6000 + 2000 * np.random.randn(n)), 0.3), 1.5)

# Halo from Brand A TV
brand_a_tv_data = np.maximum(0, 20000 + 8000 * np.sin(weeks * 2 * np.pi / 52) +
                              np.random.normal(0, 3000, n))
halo_a_tv = 0.03 * 120000 * tanh_saturation(
    geometric_adstock(brand_a_tv_data, 0.7), 1.7)

total_b = base_b + seasonal_b + own_social + own_digital + halo_a_tv

fig, ax = plt.subplots(figsize=(14, 7))

# Stacked area chart
components = [base_b, seasonal_b, own_social, own_digital, halo_a_tv]
labels = ['Base', 'Seasonality', 'Own Social', 'Own Digital', 'Halo from Brand A']
colors_stack = ['#94A3B8', '#CBD5E1', COLORS[1], COLORS[0], COLORS[4]]

# Stack from bottom up
cumulative = np.zeros(n)
for i, (comp, label, color) in enumerate(zip(components, labels, colors_stack)):
    ax.fill_between(weeks, cumulative / 1000, (cumulative + comp) / 1000,
                    alpha=0.7, color=color, label=label)
    cumulative += comp

ax.plot(weeks, total_b / 1000, color='#1E293B', linewidth=2, label='Total Revenue', linestyle='-')

ax.set_xlabel('Week')
ax.set_ylabel('Revenue ($K)')
ax.set_title('Brand B Revenue Decomposition with Halo Effects')
ax.legend(loc='upper left', fontsize=9, ncol=2)

# Annotate the halo layer
halo_pct = halo_a_tv.mean() / total_b.mean() * 100
ax.annotate(f'Halo from Brand A\n{halo_pct:.1f}% of revenue',
            xy=(60, (cumulative.mean() - halo_a_tv.mean() / 2) / 1000),
            fontsize=10, fontweight='bold', color='white',
            bbox=dict(boxstyle='round', facecolor=COLORS[4], alpha=0.85),
            ha='center')

plt.tight_layout()
plt.savefig('images/12_halo_decomposition.png', dpi=180, bbox_inches='tight')
plt.show()

print(f'\nBrand B Revenue Decomposition (weekly averages):')
print(f'  Base:                ${base_b.mean():>10,.0f} ({base_b.mean()/total_b.mean()*100:5.1f}%)')
print(f'  Seasonality:         ${seasonal_b.mean():>10,.0f} ({abs(seasonal_b.mean())/total_b.mean()*100:5.1f}%)')
print(f'  Own Social:          ${own_social.mean():>10,.0f} ({own_social.mean()/total_b.mean()*100:5.1f}%)')
print(f'  Own Digital:         ${own_digital.mean():>10,.0f} ({own_digital.mean()/total_b.mean()*100:5.1f}%)')
print(f'  Halo (Brand A TV):   ${halo_a_tv.mean():>10,.0f} ({halo_a_tv.mean()/total_b.mean()*100:5.1f}%)')
print(f'  \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'  Total:               ${total_b.mean():>10,.0f}')


Brand B Revenue Decomposition (weekly averages):
  Base:                $   179,795 ( 86.3%)
  Seasonality:         $     4,804 (  2.3%)
  Own Social:          $    15,579 (  7.5%)
  Own Digital:         $     6,805 (  3.3%)
  Halo (Brand A TV):   $     1,416 (  0.7%)
  ───────────────────────────────
  Total:               $   208,398


---

## Modeling Halo Effects with PyMC-Marketing

[PyMC-Marketing](https://github.com/pymc-labs/pymc-marketing)'s `MMM` class does not have a dedicated "halo channel" feature. However, you can model cross-brand spillover by including Brand A's spend as a `control_column` in Brand B's model. The control column captures the halo contribution without going through the media channel's adstock/saturation pipeline.

For tighter control, you can also pre-transform the halo spend (apply adstock/saturation manually as shown in this notebook) and pass the transformed values as a control.

In [12]:
from pymc_marketing.mmm import MMM, GeometricAdstock, TanhSaturation

# Halo effects: include Brand A's spend as a control in Brand B's model.
# This captures cross-brand spillover as a linear control variable.

mmm_brand_b = MMM(
    date_column="date",
    channel_columns=["brand_b_tv", "brand_b_digital"],
    control_columns=["brand_a_tv_spend"],  # <-- halo channel from Brand A
    adstock=GeometricAdstock(l_max=8),
    saturation=TanhSaturation(),
    yearly_seasonality=2,
)

print("Brand B MMM with halo from Brand A:")
print(f"  Brand B channels: {mmm_brand_b.channel_columns}")
print(f"  Control columns (halo): {mmm_brand_b.control_columns}")
print()
print("The control_columns approach treats Brand A's spend as a linear")
print("predictor in Brand B's model. For more control, you can also:")
print("  1. Pre-apply adstock/saturation to Brand A's spend (as shown above)")
print("  2. Pass the transformed values as a control column")
print("  3. Or use custom PyMC code for a shared multi-brand model")
print()
print("Trademark adjustment: reduce the brand search coefficient by applying")
print("a tighter prior (e.g., InverseGamma with smaller mean) to that channel.")

Brand B MMM with halo from Brand A:
  Brand B channels: ['brand_b_tv', 'brand_b_digital']
  Control columns (halo): ['brand_a_tv_spend']

The control_columns approach treats Brand A's spend as a linear
predictor in Brand B's model. For more control, you can also:
  1. Pre-apply adstock/saturation to Brand A's spend (as shown above)
  2. Pass the transformed values as a control column
  3. Or use custom PyMC code for a shared multi-brand model

Trademark adjustment: reduce the brand search coefficient by applying
a tighter prior (e.g., InverseGamma with smaller mean) to that channel.


---

## Summary

| Concept | Implementation | Effect |
|---------|---------------|--------|
| **Halo Effect** | Brand A's spend added as a variable in Brand B's model with a small fixed coefficient (e.g., 0.005-0.02) | Captures cross-brand spillover that traditional MMMs miss |
| **Trademark Adjustment** | Typically 10-30% reduction applied to brand search coefficient | Prevents brand search from stealing attribution from awareness channels |
| **Halo Matrix** | N x N matrix of spillover rates between all brands in portfolio | Maps the full network of cross-brand influence |
| **Budget Impact** | Halo-generating channels get higher portfolio-level ROI | Optimizer allocates more to channels that lift the whole portfolio |

**Key takeaways:**

1. Without halo modeling, you **undervalue** the brands that do heavy awareness advertising
2. Without trademark adjustment, you **overvalue** brand search at the expense of the channels that drive it
3. Halo-aware optimization gives **different and better** budget allocations for multi-brand portfolios
4. These are Simba-specific capabilities not available in PyMC-Marketing

## Next Steps

- **[Notebook 04: Portfolio Optimization](04-portfolio-optimization.ipynb)** — How halo effects feed into cross-brand budget optimization
- **[Halo Effects (Core Concepts)](../docs/core-concepts/halo-effects.md)** — Theory behind cross-brand advertising spillover
- **[Portfolio Analysis (Platform Guide)](../docs/platform-guide/portfolio-analysis.md)** — Using Simba's portfolio tools with halo-aware modeling
- **[Budget Optimization (Platform Guide)](../docs/platform-guide/budget-optimization.md)** — How the optimizer uses halo-adjusted ROI